In [ ]:
import json

with open('IMDB_top_100.json', 'r') as file:
    imdb_dataset = json.load(file)

In [31]:
vespa_feed = []

for movie in imdb_dataset:
    vespa_feed.append({
        'id': movie['id'],
        'fields': {
            'title': movie['Series_Title'],
            'description': movie['Synopsis'],
            'id': movie['id'],
            'year': movie['Released_Year'],
        }   
    })

print(vespa_feed)



[{'id': 0, 'fields': {'title': 'The Shawshank Redemption', 'description': '**The Shawshank Redemption** is a drama film that follows the story of Andy Dufresne, a banker who is sentenced to two consecutive life terms in prison for the murders of his wife and her lover, crimes he insists he did not commit. The story unfolds in Shawshank State Penitentiary over several decades, beginning in the late 1940s.\n\nOnce inside, Andy befriends Ellis "Red" Redding, the prison\'s fixer, who can smuggle in items from the outside world. Despite initial skepticism from other inmates, Andy earns their respect through his quiet resolve and intelligence. He soon becomes an asset to the corrupt warden, Samuel Norton, and the guards overseeing various illicit financial operations. Andy\'s financial expertise allows him to manage the warden\'s illegal endeavors, laundering money under a false identity.\n\nWhile facing brutal realities, including violent encounters with a gang called "The Sisters," Andy ma

In [ ]:
from vespa.application import ApplicationPackage

app_package = ApplicationPackage(name='imdbsearch')

In [ ]:
from vespa.package import Schema, Document

film_schema = Schema(name='imdb_film_schema', document=Document())

In [41]:
from vespa.package import Field

film_schema.add_fields(
    Field(
        name='title',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='description',
        type='string',
        indexing=['index', 'summary'],
        index="enable-bm25",
    ),
    Field(
        name='year',
        type='int',
        indexing=['attribute']
    ),
    Field(
        name='id',
        type='string',
        indexing=['summary']
    )
)

In [42]:
from vespa.package import FieldSet

film_schema.add_field_set(
    FieldSet(
        name='default',
        fields=['title', 'description', 'year', 'id']
    ))

In [43]:
app_package.add_schema(film_schema)

In [47]:
from vespa.deployment import VespaCloud

vespa_cloud = VespaCloud(
    tenant='akashsinghal',
    application='imdbsearch',
    application_package=app_package
)

Setting application...
Running: vespa config set application akashsinghal.imdbsearch.default
Setting target cloud...
Running: vespa config set target cloud
Success: set target to cloud in global config at /Users/akashsinghal/.vespa/config.yaml

No api-key found for control plane access. Using access token.
Checking for access token in auth.json...
Access token expired. Please re-authenticate.
Your Device Confirmation code is: ZSDM-MZBB
Automatically open confirmation page in your default browser? [Y/n] Y
Opened link in your browser:
	 https://login.console.vespa-cloud.com/activate?user_code=ZSDM-MZBB
Waiting for login to complete in browser ... done;1m⣾
Success: Logged in
 auth.json created at /Users/akashsinghal/.vespa/auth.json
Successfully obtained access token for control plane access.


In [48]:
app = vespa_cloud.deploy(region="aws-euw1-az1")

Deployment started in run 5 of dev-aws-euw1-az1 for akashsinghal.imdbsearch.default. This may take a few minutes the first time.
INFO    [12:29:48]  Deploying platform version 8.766.27 and application dev build 5 for dev-aws-euw1-az1 of default ...
INFO    [12:30:24]  Using CA signed certificate version 1
INFO    [12:30:25]  Using 1 nodes in container cluster 'imdbsearch_container'
INFO    [12:30:27]  Using 1 nodes in container cluster 'imdbsearch_container'
INFO    [12:30:35]  Session 7 for tenant 'akashsinghal' prepared and activated.
INFO    [12:30:35]  ######## Details for all nodes ########
INFO    [12:30:35]  h12249a.dev.euw1-az1.aws.vespa-cloud.net: expected to be UP
INFO    [12:30:35]  --- platform vespaai/cloud-tenant:8.766.27
INFO    [12:30:35]  --- container on port 4080 has config generation 6, wanted is 7
INFO    [12:30:35]  --- metricsproxy-container on port 19092 has config generation 7, wanted is 7
INFO    [12:30:35]  h3290h.dev.euw1-az1.aws.vespa-cloud.net: expected to

In [50]:
cert_path = app.cert
key_path = app.key

endpoint = vespa_cloud.get_mtls_endpoint(region="aws-euw1-az1")

Found mtls endpoint for imdbsearch_container
URL: https://e4aceae2.a1cf2ebf.z.vespa-app.cloud/


In [51]:
from vespa.application import Vespa

vespa_application = Vespa(endpoint, cert=cert_path, key=key_path)

In [52]:
from vespa.io import VespaResponse

def callback(response: VespaResponse, id:str):
    if not response.is_successful():
        print(f"Error when feeding document {id}: {response.get_json()}")

vespa_application.feed_iterable(vespa_feed, schema='imdb_film_schema', callback=callback)

In [53]:
from vespa.io import VespaQueryResponse

def print_hits(response: VespaQueryResponse):
    for i, hit in enumerate(response.hits):
        print(f'{i+1:>3}  {hit["fields"]['id']:>5}  {hit['fields']['title']}')

In [54]:
# Searching for 'world war' using userQuery(). This function uses the default fieldset

response = vespa_application.query(
    yql="select * from sources * where default contains 'world war'", # The default limit in Vespa is 10 results
    # query = "world war", # when using userQuery() you must provide the query as a parameter
)

print_hits(response)


  1     80  Paths of Glory
  2     50  Casablanca
  3     46  Grave of the Fireflies
  4     84  1917
  5     24  Saving Private Ryan
  6     93  Inglourious Basterds
  7     26  Life is Beautiful
  8     38  The Pianist
  9     79  Witness for the Prosecution
 10      7  Schindler's List


In [55]:
# Searching for 'world war' using userQuery(). This function uses the default fieldset

response = vespa_application.query(
    yql="select * from sources * where userQuery() limit 100", # The default limit in Vespa is 10 results
    query = "phone", # when using userQuery() you must provide the query as a parameter
)

print_hits(response)


  1     69  Memento
  2     28  The Silence of the Lambs
  3     77  High and Low
  4     56  Your Name.
